# Teilaufgabe a)

In [1]:
# Daten laden und sichten
import pandas as pd

df = pd.read_csv("../data/stock_data.csv")
df.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,2009-12-31,137.089996,137.279999,134.520004,134.520004,134.520004,4523000
1,2010-01-04,136.250000,136.610001,133.139999,133.899994,133.899994,7599900
2,2010-01-05,133.429993,135.479996,131.809998,134.690002,134.690002,8851900
3,2010-01-06,134.600006,134.729996,131.649994,132.250000,132.250000,7178800
4,2010-01-07,132.009995,132.320007,128.800003,130.000000,130.000000,11030200


In [2]:
# Daten auf die Spalte "Close" reduzieren
close = df[['Close']]
close.head()

,Close
0,134.520004
1,133.899994
2,134.690002
3,132.250000
4,130.000000


In [3]:
# Anhand von 3 unabhängigen Variablen prüfenwie wie die Tabelle aussieht, nach Einsatz der shift-Funktion
# NaN-Werte entstehen, da nach der shift-Funktion für die ersten Zeilen keine Werte mehr vorhanden sind
df_3 = pd.DataFrame()
df_3['close_t-3'] = df['Close'].shift(3)
df_3['close_t-2'] = df['Close'].shift(2)
df_3['close_t-1'] = df['Close'].shift(1)
df_3['target'] = df['Close']

df_3.head()

,close_t-3,close_t-2,close_t-1,target
0,NaN,NaN,NaN,134.520004
1,NaN,NaN,134.520004,133.899994
2,NaN,134.520004,133.899994,134.690002
3,134.520004,133.899994,134.690002,132.250000
4,133.899994,134.690002,132.250000,130.000000


In [4]:
# Funktion nimmt das ursprüngliche DataFrame und die Anzahl der Zeitfenster (window) als Eingabe und gibt ein neues DataFrame zurück, das die verschobenen Werte enthält.
# Zielvariable ist die aktuelle "Close"-Spalte, während die unabhängigen Variablen die vorherigen Werte der "Close"-Spalte sind, die durch die shift()-Funktion erstellt wurden.
def create_dataset(df, window):
    data = pd.concat(
        [df['Close'].shift(i) for i in range(window, 0, -1)] + [df['Close']],
        axis=1
    )
    data.columns = [f'close_t-{i}' for i in range(window,0,-1)] + ['target']
    # Entfernen von Zeilen mit NaN-Werten, die durch die Verschiebung entstehen
    return data.dropna()

# Erstellen von Datensätzen mit unterschiedlichen Zeitfenstern
df_3 = create_dataset(df, 3)
df_5 = create_dataset(df, 5)
df_8 = create_dataset(df, 8)
# Zur Überprüfung der Anzahl der Zeilen und Spalten in den erstellten DataFrames
print(df_3.shape)
print(df_5.shape)
print(df_8.shape)

(2258, 4)
(2256, 6)
(2253, 9)


# Teilaufgabe b)

### Modellaufbau und experimentelles Vorgehen
Für die Vorhersage des Schlusskurses am Folgetag wurde ein neuronales Netz verwendet, das auf den Schlusskursen der vorherigen Tage basiert. Je nach Experiment wurden 3, 5 oder 8 vorherige Tage als Eingabefeatures verwendet.

#### Wahl des Optimizers
Als Optimierungsverfahren wurde Adam (Adaptive Moment Estimation) eingesetzt. Adam ist ein weit verbreiteter Optimizer für neuronale Netze, da er adaptive Lernraten nutzt und sich dadurch gut für kleinere Modelle und Datensätze wie in dieser Aufgabe eignet.

#### Trainingsdauer
Initial wurde mit 150 Epochen trainiert. Dabei zeigte sich jedoch, dass nach etwa 100 Epochen kaum noch Verbesserungen der Modellleistung auftreten und teilweise sogar eine Verschlechterung des Fehlers beobachtet werden konnte. Es könnte darauf hin deuten, dass sich das Modell beginnt stärker anzupassen.

#### Erweiterung der Modellarchitektur
Begonnen wurde mit einem linearen Output-Layer. Das Ergänzen von Hidden Layers lieferte allerdings deutlich bessere Ergebnisse, über 32 - 16 - 1 konnte keine signifikante Verbesserung erzielt werden.

#### Stabilisierung des Trainings
Zu Beginn zeigte das Modell eine sehr hohe Varianz der Ergebnisse zwischen einzelnen Trainingsläufen. Um diesen Effekt systematisch zu untersuchen, wurden fünf feste Seeds verwendet. Diese Anzahl ist statistisch nicht ausreichend für eine vollständige Analyse der Modellstabilität, erlaubt jedoch eine erste Einschätzung der Streuung der Ergebnisse.

Im Anschluss wurde Early Stopping eingesetzt. Dabei wird ein Teil der Trainingsdaten als Validierungsmenge verwendet (validation_split = 0.2). Das Training wird beendet, sobald sich der Validierungsfehler über mehrere Epochen hinweg nicht mehr verbessert. Dadurch konnte die Varianz der MSE zwischend den seeds deutlich reduziert werden.
Außerdem fürhte eine Reduktion der batch size von 32 auf 16 zu etwas stabileren Ergebnissen.

#### Einfluss des Zeitfensters
Kurze Zeitfenster (3 Tage) liefern im Durchschnitt die besten Ergebnisse (MSE unter 260). Größere Fenster mit 5 oder 8 Tagen führten häufig zu höheren MSE. Das deutet darauf hin, dass weiter zurückliegende Kurse nur begrenzte zusätzliche Information für die Vorhersage liefern und teilweise eher Rauschen in das Modell einbringen. Aber auch hier muss berücksichtigt werden, dass nur 5 seeds verglichen wurden die einen Trend andeuten.


In [56]:
#  Import
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense, Input
from keras import utils
from keras.callbacks import EarlyStopping
import pandas as pd

In [57]:
# Modell erstellen
def build_model(input_dim):
    model = Sequential([
        # Input-Layer mit der Anzahl der Merkmale als Eingabedimension
        Input(shape=(input_dim,)),
        # Versteckte Schichten mit ReLU-Aktivierungsfunktion
        Dense(32, activation="relu"),
        Dense(16, activation="relu"),
        # Ausgabe-Layer mit 1 Neuron und linearer Aktivierungsfunktion
        Dense(1, activation="linear")
    ])
    # Kompilieren des Modells mit Adam-Optimizer und Mean Squared Error als Verlustfunktion
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model

In [58]:
# Funktion zum Trainieren und Evaluieren des Modells
def train_and_evaluate(df, seed, test_size=0.25, random_state=42, epochs=100, batch_size=16):
    utils.set_random_seed(seed)
    # Aufteilen der Daten in Trainings- und Testsets
    X = df.drop(columns="target")
    y = df["target"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    # Erstellen des Modells basierend auf der Anzahl der unabhängigen Variablen
    model = build_model(X_train.shape[1])
    # early stopping, um Überanpassung zu vermeiden
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        shuffle=False,
        validation_split=0.2,
        callbacks=[early_stop]
    )
    
    mse = model.evaluate(X_test, y_test, verbose=0)
    
    return mse, history

In [59]:
# Mehrere Läufe mit unterschiedlichen Seeds und Zeitfenstern
seeds = [6, 42, 56, 78, 91]
results = []

for seed in seeds:    
    for name, dataset in [("3 Tage", df_3), ("5 Tage", df_5), ("8 Tage", df_8)]:
        mse, history = train_and_evaluate(dataset, seed=seed)
        
        results.append({
            "Seed": seed,
            "Fenster": name,
            "MSE": mse
        })
 # Ergebnisse in einem DataFrame zusammenfassen und auswerten       
results_df = pd.DataFrame(results)
print(results_df.pivot(index="Seed", columns="Fenster", values="MSE"))
print(results_df.groupby("Fenster")["MSE"].agg(["mean","std","min","max"]))

Fenster      3 Tage      5 Tage      8 Tage
Seed                                       
6        240.129288  326.161438  351.779083
42       247.925613  239.326187  300.062653
56       212.704346  255.531586  227.680466
78       331.336334  347.058105  278.307098
91       258.212799  281.331879  454.474884
               mean        std         min         max
Fenster                                               
3 Tage   258.061676  44.302359  212.704346  331.336334
5 Tage   289.881839  45.810071  239.326187  347.058105
8 Tage   322.460837  86.200084  227.680466  454.474884
